# Chương 5: Đồ thị tri thức (KG) và RAG, giải thích từng hàm và chạy demo

    Notebook này chạy đúng các hàm mà demo localhost (cổng 8044, trang chủ `/`) dùng. Chạy hoàn toàn trên máy,
    không cần API key. Có bốn đường xử lý: **RAG** (tìm đoạn tài liệu rồi để mô hình viết), **công cụ thống kê**
    (pandas trên dữ liệu CDC BRFSS 2023), **máy tính** và **tra đồ thị tri thức**. Bài học chính: câu hỏi kiến thức
    đi qua RAG, còn câu hỏi cần con số chính xác phải đi qua công cụ, vì mô hình ngôn ngữ nhỏ có thể sai.

In [1]:
# Cell KG-01
import os, sys, json, time, inspect, copy
from pathlib import Path
for key in ['OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS']:
    os.environ[key] = '2'
os.environ['KERAS_BACKEND'] = 'tensorflow'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p/'src/tieuluan').exists())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
OUT = ROOT/'results/tieuluan/complete/notebooks'
OUT.mkdir(parents=True, exist_ok=True)
print('Project:', ROOT)

Project: E:\WINDOW\Project\intelligent-system-assignment-01


## 1. Dữ liệu và mô hình đang dùng

    - **Kho tài liệu tiếng Anh**: 183 đoạn từ bốn trang NIDDK về bệnh tiểu đường (`diabetes_corpus.jsonl`) và 9.177 đoạn
      từ các báo cáo tài chính FinQA (`finance_corpus.jsonl`).
    - **Dữ liệu thống kê mới**: CDC BRFSS 2023 (khảo sát sức khỏe người trưởng thành Hoa Kỳ), mẫu ngẫu nhiên 150.000
      người trả lời hợp lệ, lưu CSV ở `data/chapter05/brfss2023_diabetes_150k.csv`. Dữ liệu cũ (UCI 1999–2008) không còn được dùng.
    - **Bộ sinh câu trả lời**: `Qwen/Qwen2.5-1.5B-Instruct` chạy trên CPU (tải bằng `scripts/chapter05/download_llm.py`
      vào `models/chapter05/llm`). Bộ sinh cũ flan-t5-small chỉ cho ra vài từ nên được thay; lớp `LocalGenerator` cũ vẫn còn trong mã.

In [2]:
# Cell KG-02
manifest = json.loads((ROOT/'data/chapter05/brfss2023_manifest.json').read_text(encoding='utf8'))
print({k: manifest[k] for k in ['source','year','raw_rows','valid_rows','sampled_rows','license']})
survey = pd.read_csv(ROOT/'data/chapter05/brfss2023_diabetes_150k.csv')
print('Số dòng:', len(survey), '| Số cột:', survey.shape[1])
display(survey.head(5))
display(survey.diabetes_status.value_counts().rename('số người').to_frame())
llm_name = (ROOT/'models/chapter05/llm/SOURCE.txt').read_text(encoding='utf8').strip()
print('Mô hình sinh:', llm_name)

{'source': 'CDC BRFSS 2023 (LLCP2023XPT.zip)', 'year': 2023, 'raw_rows': 433323, 'valid_rows': 280420, 'sampled_rows': 150000, 'license': 'US government public data (public domain)'}


Số dòng: 150000 | Số cột: 15


,diabetes_status,age_group,sex,bmi,bmi_category,physical_activity,high_blood_pressure,high_cholesterol,smoker_status,general_health,heart_disease,stroke,income_group,education,state
0,no,25-29,male,23.40,normal,1,1,0,never,4,0,0,50-100k,college_graduate,Michigan
1,no,40-44,female,30.41,obese,1,0,0,never,2,0,0,50-100k,college_graduate,Utah
2,yes,50-54,male,34.97,obese,0,0,0,never,4,0,1,50-100k,some_college,South Dakota
3,no,35-39,male,51.54,obese,1,1,1,former,2,0,0,15-25k,less_than_high_school,Indiana
4,no,75-79,female,21.03,normal,1,0,0,never,2,0,0,35-50k,college_graduate,Vermont


,số người
diabetes_status,
no,122896
yes,22101
prediabetes,3898
gestational_only,1105


Mô hình sinh: Qwen/Qwen2.5-1.5B-Instruct


## 2. Đồ thị tri thức: chuẩn hóa → tạo nút → tạo cạnh → truy vấn

    `build_document_graph(chunks, domain)` nhận các đoạn đã có metadata và trả về NetworkX DiGraph. Ví dụ dưới dùng một
    dòng giả định để dễ kiểm tra (không phải báo cáo thật). Sau đó nạp đồ thị tài chính thật (công ty → báo cáo → đoạn văn)
    và tra công ty V (Visa) để thấy câu hỏi "công ty này có những báo cáo nào" trả lời được bằng đồ thị mà không cần mô hình.

In [3]:
# Cell KG-03
from src.chapter04.knowledge import build_document_graph, safe_calculate, load_graph
chunks = [{'id':'demo:row1','company':'DEMO','report':'DEMO:2023',
    'year':'2023','filename':'demo-report','kind':'table',
    'text':'Revenue 2022: 100 USD; 2023: 120 USD',
    'source_url':'https://example.org/demo'}]
graph = build_document_graph(chunks,'finance')
print('Nút:',list(graph.nodes))
print('Cạnh:',list(graph.edges(data=True)))
print('Nguồn cha:',list(graph.predecessors('passage:demo:row1')))
print(inspect.getsource(build_document_graph))
fin = load_graph(ROOT/'knowledge_graph/chapter04/finance_graph.json.gz')
kinds = pd.Series([d['type'] for _, d in fin.nodes(data=True)]).value_counts()
display(kinds.rename('số nút').to_frame())
reports = [fin.nodes[n] for n in fin.successors('company:V')]
print('Công ty V có', len(reports), 'báo cáo:', sorted(r['year'] for r in reports))

Nút: ['company:DEMO', 'report:DEMO:2023', 'passage:demo:row1']
Cạnh: [('company:DEMO', 'report:DEMO:2023', {'relation': 'PUBLISHED'}), ('report:DEMO:2023', 'passage:demo:row1', {'relation': 'HAS_PASSAGE'})]
Nguồn cha: ['report:DEMO:2023']
def build_document_graph(chunks,domain):
    graph=nx.DiGraph(domain=domain)
    for chunk in chunks:
        if domain=='finance':
            company='company:'+chunk['company'];report='report:'+chunk['report']
            graph.add_node(company,type='Company',ticker=chunk['company'])
            graph.add_node(report,type='Report',year=chunk['year'],filename=chunk['filename'],source_url=chunk['source_url'])
            graph.add_edge(company,report,relation='PUBLISHED')
        else:
            report='document:'+chunk['source_url']
            graph.add_node(report,type='Document',source_url=chunk['source_url'])
        node='passage:'+chunk['id']
        graph.add_node(node,type='TableRow' if chunk.get('kind')=='table' else 'Passage',chunk_id=ch

,số nút
Passage,7276
TableRow,1901
Report,299
Company,97


Công ty V có 2 báo cáo: ['2008', '2013']


## 3. RRF: hợp thứ hạng thay vì cộng điểm khác thang đo

    Truy hồi lai kết hợp BM25 (khớp từ khóa) với vector (khớp nghĩa). `reciprocal_rank_fusion` nhận nhiều danh sách id và
    cộng 1/(60+hạng). Ví dụ hai danh sách dưới giúp tự tính điểm của B. Hàm chỉ trả về id theo điểm giảm dần.

In [4]:
# Cell KG-04
from src.chapter04.retrieval import reciprocal_rank_fusion
rankings = [['A','B','C'],['B','D','A']]
print('Hạng hợp nhất:',reciprocal_rank_fusion(rankings,k=4))
print('Điểm B:',1/62+1/61)
print(inspect.getsource(reciprocal_rank_fusion))

Hạng hợp nhất: ['B', 'A', 'D', 'C']
Điểm B: 0.03252247488101534
def reciprocal_rank_fusion(rankings,k=5,c=60):
    scores=defaultdict(float)
    for ranking in rankings:
        for rank,cid in enumerate(ranking,1):scores[cid]+=1/(c+rank)
    return sorted(scores,key=lambda cid:(-scores[cid],cid))[:k]



## 4. Một lượt RAG hoàn chỉnh bằng câu hỏi tiếng Việt

    `RAGEngine.answer` làm bốn việc: (1) **dịch** câu hỏi tiếng Việt thành truy vấn tiếng Anh vì kho tài liệu bằng tiếng Anh;
    (2) **truy hồi lai** ba đoạn gần nhất; (3) **mô hình viết** câu trả lời chỉ dựa trên ba đoạn đó (viết tiếng Anh trước vì mô hình
    nhỏ đọc tiếng Anh tốt hơn, rồi dịch sang tiếng Việt bằng bảng thuật ngữ); (4) đóng gói nguồn và trạng thái.
    Kết quả luôn có trạng thái `generated_unverified`: các mã nguồn tồn tại nhưng chưa ai kiểm chứng câu viết có đúng ý nguồn hay không.
    Lần chạy này mất khoảng nửa phút đến vài phút vì mô hình chạy trên CPU; ô sau in thời gian thực đo.

In [5]:
# Cell KG-05
from deployment.chapter04.app import rag, agent
engine = rag()
t0 = time.perf_counter(); engine.generator.load(); print('Nạp mô hình sinh: %.1f giây' % (time.perf_counter()-t0))
question = 'Xét nghiệm A1C cho biết điều gì?'
answer = engine.answer(question,'diabetes')
print('Câu hỏi        :', question)
print('Truy vấn tiếng Anh:', answer['query_used'])
print('Bản tiếng Anh  :', answer['english_answer'])
print('Câu trả lời    :', answer['answer'])
print('Trạng thái     :', answer['status'], '| thời gian %.1f giây' % answer['seconds'])
display(pd.DataFrame([{'id':e['id'],'văn bản':e['text'][:160]} for e in answer['evidence']]))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Nạp mô hình sinh: 7.5 giây


Câu hỏi        : Xét nghiệm A1C cho biết điều gì?
Truy vấn tiếng Anh: What does an A1C test indicate?
Bản tiếng Anh  : An A1C test indicates your average blood glucose level over the last 3 months.
Câu trả lời    : Xét nghiệm A1C cho thấy mức đường huyết trung bình trong 3 tháng gần nhất.
Trạng thái     : generated_unverified | thời gian 38.9 giây


,id,văn bản
0,managing-diabetes:21,The A1C test shows your average blood glucose ...
1,tests-diagnosis:41,The A1C test is a blood test that provides you...
2,tests-diagnosis:43,Your doctor will report your A1C test result a...


## 5. Khi tài liệu không có câu trả lời: từ chối thay vì bịa

    Có hai lớp bảo vệ. Lớp một là `lexical_support`: nếu không từ khóa nội dung nào của truy vấn xuất hiện trong ba đoạn tìm được
    thì dừng ngay, không gọi mô hình. Lớp hai là chính mô hình, được dặn trả lời đúng một từ `INSUFFICIENT` khi các đoạn không chứa đáp án.
    Hai lớp này không hoàn hảo, nhưng làm giảm nguy cơ mô hình bịa.

In [6]:
# Cell KG-06
from src.chapter04.rag import lexical_support, content_tokens
refused = engine.answer('Giá vàng hôm nay là bao nhiêu?','diabetes')
print('Truy vấn:', refused['query_used'], '| từ khóa nội dung:', content_tokens(refused['query_used']))
print('Trạng thái:', refused['status'], '|', refused['answer'])
assert refused['status'] == 'insufficient_evidence'

Truy vấn: What is the current price of gold? | từ khóa nội dung: ['current', 'price', 'gold']
Trạng thái: insufficient_evidence | Không đủ thông tin trong tài liệu để trả lời câu hỏi này.


## 6. Công cụ thống kê: hiểu nhiều dạng câu hỏi tự nhiên

    `DiabetesStats` nhận ý định bằng luật và từ khóa (đã bỏ dấu để khớp cả câu không dấu): đếm, tỉ lệ, so sánh hai nhóm,
    phân theo nhóm, BMI trung bình, kèm bộ lọc như nhóm tuổi, giới, BMI, huyết áp cao, hút thuốc, bang. Mọi con số tính bằng pandas
    nên chính xác theo dữ liệu. Câu không hiểu được trả lời lịch sự kèm danh sách câu mẫu, không báo lỗi.

In [7]:
# Cell KG-07
from src.chapter04.stats import DiabetesStats
stats = DiabetesStats(ROOT/'data/chapter05/brfss2023_diabetes_150k.csv')
questions = ['Có bao nhiêu người trong dữ liệu khảo sát?',
             'Tỉ lệ người mắc tiểu đường là bao nhiêu?',
             'So sánh tỉ lệ tiểu đường giữa nam và nữ',
             'BMI trung bình của người bị tiểu đường là bao nhiêu?',
             'Tỉ lệ tiểu đường ở người huyết áp cao',
             'What percentage of people over 65 have diabetes?',
             'Diabetes rate in Texas']
rows = []
for q in questions:
    r = stats.answer(q); rows.append({'câu hỏi': q, 'câu trả lời': r['answer']})
display(pd.DataFrame(rows))
table = stats.answer('Tỉ lệ tiểu đường theo nhóm tuổi')['table']
display(pd.DataFrame(table['rows'], columns=table['columns']))
print(stats.answer('Hôm nay trời đẹp'))
# Kiểm chứng độc lập bằng pandas thuần:
assert stats.answer('Tỉ lệ người mắc tiểu đường là bao nhiêu?')['value'] == (survey.diabetes_status == 'yes').mean()*100

,câu hỏi,câu trả lời
0,Có bao nhiêu người trong dữ liệu khảo sát?,Dữ liệu có 150.000 người trả lời khảo sát BRFS...
1,Tỉ lệ người mắc tiểu đường là bao nhiêu?,"Trong 150.000 người được khảo sát năm 2023, 14..."
2,So sánh tỉ lệ tiểu đường giữa nam và nữ,So sánh tỉ lệ người được chẩn đoán tiểu đường ...
3,BMI trung bình của người bị tiểu đường là bao ...,BMI trung bình của người được chẩn đoán tiểu đ...
4,Tỉ lệ tiểu đường ở người huyết áp cao,"Trong 64.924 người bị huyết áp cao năm 2023, 2..."
5,What percentage of people over 65 have diabetes?,"Among 58,735 respondents (age 65 and over) in ..."
6,Diabetes rate in Texas,"Among 3,384 respondents (in Texas) in 2023, 16..."


,Nhóm tuổi,Tỉ lệ,Số ca,Số người
0,18-24 tuổi,"1,5%",77,5.057
1,25-29 tuổi,"2,1%",122,5.772
2,30-34 tuổi,"3,3%",257,7.731
3,35-39 tuổi,"4,5%",434,9.552
4,40-44 tuổi,"6,8%",713,10.492
5,45-49 tuổi,"10,3%",1.078,10.483
6,50-54 tuổi,"14,1%",1.731,12.307
7,55-59 tuổi,"16,4%",2.209,13.432
8,60-64 tuổi,"18,4%",3.031,16.439
9,65-69 tuổi,"20,9%",3.726,17.821


None


## 7. Bộ định tuyến công cụ (ToolAgent): chọn đúng đường cho từng câu

    `ToolAgent.route` đọc câu hỏi và chọn: có phép tính hai số thì dùng máy tính; có tên công ty kèm từ "báo cáo" thì tra đồ thị;
    có từ khóa thống kê thì dùng công cụ thống kê; còn lại là RAG. Đây là bộ định tuyến luật, không phải agent LLM tự lập kế hoạch.
    Ô dưới chạy các đường không cần mô hình sinh để thấy câu trả lời bằng văn xuôi cùng "cách agent xử lý".

In [8]:
# Cell KG-08
worker = agent()
rows = []
for q, dom in [('Tính tăng trưởng doanh thu từ 100 lên 130','finance'),
               ('250 chia 5','finance'),
               ('Công ty V có những báo cáo nào?','finance'),
               ('So sánh tỉ lệ tiểu đường giữa nam và nữ','diabetes')]:
    r = worker.ask(q, dom)
    rows.append({'câu hỏi': q, 'đường xử lý': r['route'], 'câu trả lời': r['answer'], 'nhãn tin cậy': r['confidence']['kind']})
display(pd.DataFrame(rows))
print(worker.ask('So sánh tỉ lệ tiểu đường giữa nam và nữ','diabetes')['steps'])
print(safe_calculate('growth',[100,120],['USD','USD']))
for operation, values, units in [('divide',[1,0],['USD','USD']), ('growth',[100,120],['USD','VND'])]:
    try:
        safe_calculate(operation,values,units)
    except ValueError as exc:
        print('Từ chối đúng:',exc)
    else:
        raise AssertionError('Đầu vào sai không bị chặn')

,câu hỏi,đường xử lý,câu trả lời,nhãn tin cậy
0,Tính tăng trưởng doanh thu từ 100 lên 130,calculate,"Từ 100 lên 130 là tăng 30%, tức +30 so với số ...",computed
1,250 chia 5,calculate,250 ÷ 5 = 50.,computed
2,Công ty V có những báo cáo nào?,graph,"Công ty V có 2 báo cáo trong kho: 2008, 2013. ...",lookup
3,So sánh tỉ lệ tiểu đường giữa nam và nữ,statistics,So sánh tỉ lệ người được chẩn đoán tiểu đường ...,computed


[{'title': 'Nhận ra đây là câu hỏi về số liệu', 'detail': 'Câu có từ như "bao nhiêu", "tỉ lệ", "so sánh", "trung bình" nên đi qua công cụ thống kê, không qua mô hình sinh.'}, {'title': 'Hiểu câu hỏi', 'detail': 'Công cụ hiểu câu hỏi là: tính tỉ lệ phần trăm, của người được chẩn đoán tiểu đường, so sánh theo giới tính.'}, {'title': 'Dùng pandas tính trên 150.000 dòng CSV', 'detail': 'Lọc các dòng phù hợp rồi đếm hoặc lấy trung bình; mọi số lấy trực tiếp từ dữ liệu.'}, {'title': 'Viết kết quả thành câu văn', 'detail': 'Kèm bảng nhỏ nếu so sánh hoặc phân nhóm.'}]
{'value': 20.0, 'formula': '(120.0 - 100.0) / 100.0 × 100%', 'unit': '%'}
Từ chối đúng: Division by zero
Từ chối đúng: Units must match


## 8. Đọc kết quả thực nghiệm đã lưu

    Các bảng này đọc từ JSON của thí nghiệm truy hồi trên FinQA ở bài Chương 4, không chạy lại benchmark. Chúng được đo với bộ sinh cũ
    (flan-t5-small), nên cột sinh văn bản không áp dụng cho Qwen. So sánh global với oracle_report phải ghi đúng điều kiện.
    Câu hỏi để giải thích: Hit@5, Recall@5, Complete@5 khác nhau thế nào?

In [9]:
# Cell KG-09
result = json.loads((ROOT/'results/chapter04/finance_rag_results.json')
                    .read_text(encoding='utf8'))
display(pd.DataFrame(result['retrieval'])[
    ['method','scope','hit_at_5','recall_at_5','complete_at_5']])
(OUT/'kg_demo.json').write_text(json.dumps({
    'question':question,'answer':answer['answer'],'english_answer':answer['english_answer'],
    'query_used':answer['query_used'],'status':answer['status'],'model':answer['model'],
    'evidence_ids':[e['id'] for e in answer['evidence']],
    'survey_rows':len(survey)},
    ensure_ascii=False,indent=2),encoding='utf8')

,method,scope,hit_at_5,recall_at_5,complete_at_5
0,bm25_raw,global,0.519819,0.403380,0.305776
1,bm25_raw,oracle_report,0.809740,0.676948,0.552661
2,bm25_headers,global,0.619479,0.509377,0.408834
3,bm25_headers,oracle_report,0.894677,0.783794,0.673839
4,dense,global,0.621744,0.505015,0.395243
5,dense,oracle_report,0.913930,0.798022,0.671574
6,hybrid,global,0.682899,0.560717,0.448471
7,hybrid,oracle_report,0.926387,0.819974,0.705549


501

## Giải thích từng hàm khi bảo vệ

    `build_document_graph`: tạo schema và cạnh có nguồn. `RAGEngine.retrieve`: chọn đoạn (BM25 + vector + RRF).
    `RAGEngine.to_english_query`: dịch truy vấn. `ChatGenerator.generate_full`: viết tiếng Anh từ ba đoạn rồi dịch.
    `lexical_support`: chặn trường hợp không có từ khóa nào khớp. `DiabetesStats.answer`: nhận ý định rồi tính bằng pandas.
    `safe_calculate`: tính số theo toán hạng cấu trúc, không dùng eval. `ToolAgent.route`: chọn công cụ.
    RAG không bảo đảm hết ảo giác; chỉ số liệu do công cụ tính mới đáng tin tuyệt đối.